# Connecting Glosses and References

In [ ]:
## IMPORTANT: Because of kraken, we need Python < 3.13
import sys  # system-specific parameters and functions

assert(sys.version_info[0] == 3 and sys.version_info[1] < 13)

# install the necessary python packages
!pip install -r ./XMLTools/requirements.txt

In [1]:

import matplotlib.pyplot as plt  # draw images
import matplotlib.patches as patches  # drawing polygons
import numpy as np  # matrix manipulation
import sys  # system parameters etc.
sys.path.append("./XMLTools") 

# Kraken OCR

# Local modules
from XMLTools.glossit_dataclasses import LineType
from XMLTools.xml_extraction import METSBook

# 1. Test the word BB recognition for a single page

In [2]:
mets_path = "./XMLTools/pages/mehrzeilige_glossen/METS.xml"
tei_path = "./XMLTools/pages/mehrzeilige_glossen/first.xml"
ocr_model_path = "./XMLTools/AngBerKasSgBN1616.mlmodel"

#mets_path = "./download/METS.xml"
#tei_path = "./download/BCr_first.xml"
#ocr_model_path = "./XMLTools/irish_minuscule_fourth_train_60pcr.mlmodel"

book = METSBook(mets_path=mets_path, tei_path=tei_path, ocr_model_path=ocr_model_path) 
page = book[0]

fig, ax = plt.subplots(1, 2, figsize=(20, 30))
ax[0].axis("off")
ax[1].axis("off")
ax[0].imshow(np.asarray(page.pageimg))
ax[1].imshow(np.asarray(page.pageimg))

for line in page.get_main_text_lines() + page.get_gloss_lines():
    color = "r"
    if line.type == LineType.REFERENCE_SIGN:
        color = "b"
    elif line.type == LineType.INTERLINEAR_LINE_GLOSS:
        color = "g"
    elif line.type == LineType.INTERLINEAR_LINE_SIGNE_DE_RENVOI:
        color = "m"
    elif line.type == LineType.INTERLINEAR_LINE_CORRECTION:
        color = "c"
    elif line.type == LineType.INTERLINEAR_LINE_ADDITION:
        color = "k"
        
    for word_coordinate in line.word_bounding_boxes:
        rect = patches.Polygon(word_coordinate, linewidth=1, edgecolor=color, facecolor='none')
        ax[1].add_patch(rect)

/home/tristan/.glossit/lib/python3.12/site-packages/kraken/rpred.py:335: UserWarning: Using legacy polygon extractor, as the model was not trained with the new method. Please retrain your model to get speed improvement.
  warnings.warn('Using legacy polygon extractor, as the model was not trained with the new method. Please retrain your model to get speed improvement.')


KeyboardInterrupt: 

In [2]:
import pickle

with open("./XMLTools/project.glp", "rb") as file_handle:
    contents = pickle.load(file_handle)

In [3]:
from gui_files.program_state import _ProgramState
state = _ProgramState()
state.from_dict(contents)

In [20]:
with open("./XMLTools/test.pkl", "rb") as file_handle:
    connectors = pickle.load(file_handle)

In [23]:
print(connectors.connection_chains)

[[<glossit_connect_glosses.ConnectedPair object at 0x7e72966419d0>, <glossit_connect_glosses.ConnectedPair object at 0x7e72966428d0>, <glossit_connect_glosses.ConnectedPair object at 0x7e7296641ac0>]]


In [18]:
from gui_files.graphics import *
print(state.per_page_connections[state.current_page_index].connections)
print(state.current_page_index)

[<glossit_connect_glosses.ConnectedPair object at 0x7e7299230170>, <glossit_connect_glosses.ConnectedPair object at 0x7e72992bb680>, <glossit_connect_glosses.ConnectedPair object at 0x7e72991bf1d0>]
0


# 2. Connect Glosses

In [3]:

import difflib  # matching strings
import sys  # system parameters etc.

# Kraken OCR

sys.path.append("./XMLTools") 

# Local modules
from XMLTools.xml_extraction import METSBook

from XMLTools.glossit_connect_glosses import *

mets_path = "./XMLTools/pages/mehrzeilige_glossen/METS.xml"
tei_path = "./XMLTools/pages/mehrzeilige_glossen/first.xml"
ocr_model_path = "./XMLTools/AngBerKasSgBN1616.mlmodel"

book = METSBook(mets_path=mets_path, tei_path=tei_path, ocr_model_path=ocr_model_path, verbose=True)
#book = METSBook(mets_path=mets_path, xslt_path=xslt_path) 
page = book[0]

connector = GlossOnPageConnector(page)



Getting word coordinates for each page:


  0%|                                                     | 0/1 [00:00<?, ?it/s]/home/tristan/.glossit/lib/python3.12/site-packages/kraken/rpred.py:335: UserWarning: Using legacy polygon extractor, as the model was not trained with the new method. Please retrain your model to get speed improvement.
  warnings.warn('Using legacy polygon extractor, as the model was not trained with the new method. Please retrain your model to get speed improvement.')
100%|████████████████████████████████████████████| 1/1 [01:49<00:00, 109.60s/it]


In [4]:

def quick_build_pair(page, line_id_1, line_id_2):
    return ConnectedPair(page.get_object_from_id(line_id_1), page.get_object_from_id(line_id_2))

def quick_build_cycle(page, id_list):
    cycle = []
    for idx in range(len(id_list)-1):
        cycle.append(quick_build_pair(page, id_list[idx], id_list[idx+1]))
    return cycle

In [6]:
connections = quick_build_cycle(page, [
    "eSc_line_a2269548",
    "eSc_line_51d77c47",
    "eSc_line_58a219fc",
    "eSc_line_c50a8fcb",
    "eSc_line_da20b4de",
    "eSc_line_ae063059",
    "eSc_line_d90e3870",
    "eSc_line_338f6933",
    "eSc_line_3dc3ee51",
    "eSc_line_b9413214",
    "eSc_line_1c990ac5",  # ref
    "eSc_line_d2c801de",  # ref
])

connections.append(ConnectedPair(page.get_object_from_id("eSc_line_d2c801de"), Word(line=page.get_object_from_id("eSc_line_6545c663"), word_idx=5)))
chain = [connections]

In [8]:
tei = connector.apply_connections(chain)
with open ("./XMLTools/pages/mehrzeilige_glossen/first2.xml", "w") as file_handle:
    file_handle.write(str(tei))